# Lesson 16 Lab — TensorRT INT4 Block Quantization: Q/DQ, Packing, and WoQ

**Puzzle:** What must be present in a graph and serialized weight buffer before TensorRT can consume INT4 weights?

This notebook keeps the RTX 5090 outputs from a complete run. Read the theory cells, make a prediction, and then use **Run All** on your own GPU.


## Why this matters

TensorRT INT4 is not merely a tensor cast. The graph must express quantize/dequantize semantics, weights must use supported per-block scales, and signed four-bit codes must be packed two per byte in the expected order. A correct reference packer is a prerequisite, not evidence that an engine was built.


## 0. Predict before running

1. Write the signed INT4 code range and calculate packed bytes for a 512×1024 matrix.
2. Predict the metadata and error implications of block size 64.
3. Separate Q/DQ correctness, packing correctness, engine build, operator trace, and timing into distinct gates.

For each answer, name the observation that would prove you wrong.


## 1. Name the concrete objects

TensorRT explicit quantization represents quantization choices with Q/DQ semantics and consumes packed low-bit weights plus scales under supported block/layout constraints.

- Explicit quantization represents scale decisions with Quantize/Dequantize semantics.
- Signed INT4 codes occupy two nibbles per byte when packed.
- TensorRT support has specific block-size and placement rules that a generic fake-quant experiment cannot prove.


## 2. Derive the mechanism

For signed INT4, two 4-bit two's-complement codes occupy one byte. Block Q/DQ applies one scale to a supported group, reconstructing floating-point values for the consuming operation or enabling a fused weight-only implementation.

For TensorRT-style symmetric INT4, codes lie in `[-8,7]` and dequantization multiplies by a per-block scale. Two four-bit two's-complement nibbles fit in one byte; unpacking must restore sign correctly. With 524,288 weights, ideal packed code storage is 262,144 bytes before scales and alignment.

Graph Q/DQ nodes preserve the scale decision across export and allow the compiler to place quantized boundaries. TensorRT currently treats INT4 as weight-only and constrains block sizes/axes. A Python Q/DQ tensor can test the math, but only a serialized engine and inspected layer implementation establish TensorRT execution.


## 3. Verify the execution environment

The next cell asserts CUDA availability, fixes the seed, locates the lesson, and prints a sanitized GPU/PyTorch/CUDA record. Check it before interpreting output.


In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "16-tensorrt-int4"
device = require_cuda()
torch.manual_seed(2026 + 16)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 4. Freeze the comparison

| Role | This run |
|---|---|
| Baseline | floating-point 512×1024 weight tensor |
| Candidate | block-64 INT4 Q/DQ plus explicit nibble pack/unpack |
| Held constant | weight tensor, grouping axis, scale rule, code order, CUDA numerical reference |
| Measurements | packed bytes, exact code round-trip, RMSE/cosine, TensorRT package probe |
| Evidence | `pytorch-gpu` |

**Experiment:** Perform block INT4 Q/DQ and nibble packing on CUDA, verify exact unpacking, and separately probe the TensorRT package.


## 5. Read the experiment code

The CUDA lab validates block Q/DQ and exact nibble round-trip while an independent package probe prevents a false TensorRT-engine claim.

The notebook quantizes blocks, packs adjacent signed codes into low/high nibbles, unpacks them, restores sign, and asserts exact equality with the original codes. It then dequantizes for error measurement. A separate import probe records whether TensorRT is available.

This ordering distinguishes serialization bugs from numerical loss. Exact code round-trip is necessary even when dequantized RMSE looks plausible, because a nibble-order or sign bug can be masked by aggregate statistics.

Only after these variables match the protocol should the cell be executed.


In [2]:
import importlib.util
w=torch.randn(512,1024,device=device); q,scales,dq=symmetric_quantize(w,bits=4,group_size=64)
codes=(q.to(torch.int16)&0xF).flatten(); packed=(codes[0::2]|(codes[1::2]<<4)).to(torch.uint8)
lo=(packed.to(torch.int16)&0xF); hi=((packed.to(torch.int16)>>4)&0xF); unpack=torch.stack([lo,hi],1).flatten()
unpack=torch.where(unpack>=8,unpack-16,unpack).to(torch.int8).reshape_as(q)
result=base_result(16,"pytorch-gpu"); result.update({"shape":list(w.shape),"group_size":64,"packed_bytes":packed.numel(),
    "codes_exact_after_unpack":bool(torch.equal(q,unpack)),"qdq_error":error_metrics(w,dq),
    "tensorrt_installed":importlib.util.find_spec("tensorrt") is not None,
    "conclusion":"Block Q/DQ and nibble packing were validated; TensorRT engine execution was not inferred from the reference path."})


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Weight shape | 512 × 1024 |
| Group size | 64 |
| Packed code bytes | 262,144 bytes |
| Exact pack/unpack | yes |
| Q/DQ RMSE | 0.107706 |
| TensorRT installed | no |


## 7. Interpret rather than merely print

The 512×1024 matrix produced exactly 262,144 packed bytes, and every code survived pack/unpack. Block-64 Q/DQ yielded RMSE 0.107706 and cosine 0.994257. TensorRT was not installed, so no engine, TensorRT layer, or latency result exists.

The outcome validates a semantic reference and serialized code layout. It does not validate TensorRT's supported axis rules for a concrete ONNX graph or the performance of an INT4 WoQ kernel.

**Inspection rule:** Packing correctness and Q/DQ error are real; engine build and latency remain unmeasured unless TensorRT is installed and executes.


## 8. Keep the evidence label honest

This run is labeled **`pytorch-gpu`**. The measured tensors and operations ran on CUDA through PyTorch. The result does not name a separate production backend unless an operator trace identifies it.

The next cell writes the complete structured result; its existing saved output is part of the checked-in evidence.


In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "codes_exact_after_unpack": true,
  "conclusion": "Block Q/DQ and nibble packing were validated; TensorRT engine execution was not inferred from the reference path.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "pytorch-gpu",
  "executed_at_utc": "2026-08-07T14:45:49+00:00",
  "group_size": 64,
  "lesson": 16,
  "packed_bytes": 262144,
  "qdq_error": {
    "cosine": 0.99425673,
    "mae": 0.09142464,
    "max_abs": 0.32329714,
    "rmse": 0.10770572
  },
  "schema_version": 1,
  "shape": [
    512,
    1024
  ],
  "tensorrt_installed": false
}
Saved: artifacts/rtx5090-result.json


## 9. Make the bounded decision

> Validate graph semantics, packing, scales, engine inspection, and timing as separate gates.

**Acceptance/rollback:** Round-trip every packed code, verify scale axis/block size and ONNX Q/DQ placement, inspect the built engine, then benchmark the engine against the same baseline.

**Failure analysis:** Mistakes include treating unsigned nibbles as signed values, reversing low/high order, dropping scale layout, or claiming 0.5 byte per weight without metadata and padding. A successful engine build can still insert dequantize work that defeats the expected benefit, so engine inspection is required.


## 10. Extend the evidence

Export a minimal Q/DQ ONNX graph with block size 64, build it under a pinned TensorRT version, inspect the engine layers, and compare outputs with the reference packer. Then profile latency and memory for several M dimensions to find where WoQ becomes beneficial.

The full derivation, reproduction command, evidence boundary and primary references are in [`README.md`](README.md).
